### Stacking algorithm -> classification models -> using separate training data for each part of the stacking algorithm

**Experimental, needs more testing whether this approach actually works well**

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import seaborn as sns
from imblearn.under_sampling import TomekLinks
from sklearn.ensemble import IsolationForest
from scipy import stats
from sklearn.preprocessing import label_binarize

In [6]:
df = pd.read_csv("credit_group_classification_processed.csv")

In [7]:
# X/y split etc.
target = "Credit_Score"

# categorical features need to be separated for certain algorithms, like CatBoost
categorical_features = ['Credit_Mix', 'Payment_of_Min_Amount_No', 'Payment_of_Min_Amount_Yes', 'Payment_Behaviour_High_spent_Large_value_payments', 'Payment_Behaviour_High_spent_Medium_value_payments', 'Payment_Behaviour_High_spent_Small_value_payments', 'Payment_Behaviour_Low_spent_Large_value_payments', 'Payment_Behaviour_Low_spent_Medium_value_payments']

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# SCALING => some of the algorithms require this
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Stacking Classifier -> we use XGBoost + RF + LGBM again

#### Here we create a custom wrapper for the StackingClassifier in order to incorporate a different subset of data for each model

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin

# in order to use a different set of data for each algorithm in the stacking algorithm
# we have to override the default fit-function
class CustomStackingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators, final_estimator, cv=5):
        self.estimators = estimators
        self.final_estimator = final_estimator
        self.cv = cv

    # OVERRIDE THE DEFAULT FIT FUNCTION
    def fit(self, X, y):
        # process through each model in the stacking algorithm
        for name, model in self.estimators:
            # get a random 75% subset of training data
            X_train, _, y_train, _ = train_test_split(X, y, test_size=0.25)

            # train this particular algorithm with this version of data
            model.fit(X_train, y_train)

        # handle the meta model
        meta_features = self._get_meta_features(X)
        self.final_estimator.fit(meta_features, y)
        return self

    # override also the predict -function
    def predict(self, X):
        meta_features = self._get_meta_features(X)
        return self.final_estimator.predict(meta_features)
    
    # override also the predict_proba -function
    def predict_proba(self, X):
        meta_features = self._get_meta_features(X)
        return self.final_estimator.predict_proba(meta_features)

    # override also the meta features
    def _get_meta_features(self, X):
        meta_features = np.column_stack([model.predict(X) for name, model in self.estimators])
        return meta_features

# meta model is the judge or optimizer of the stacking algorithm
# => it assigns weights to the models above based on their performance
# meta_model = LogisticRegression()
meta_model = cb.CatBoostClassifier(verbose=0)

# stacking regression algorithm -> USE THE CUSTOM CLASSIFIER FROM ABOVE
stacking_model = CustomStackingClassifier(
    estimators=[('xgb', xgb.XGBClassifier(enable_categorical=True, objective='multi:softprobar', num_class=len(np.unique(y_train)))),
                ('randomforest', RandomForestClassifier(n_estimators=100)),
                ('lgbm', lgb.LGBMClassifier(objective='multiclass'))
                ],
    final_estimator = meta_model,
    cv=5
)

# fit the model
stacking_model.fit(X_train, y_train)

# test predictions and metrics
predictions = stacking_model.predict(X_test)

# couple of metrics
accuracy = stacking_model.score(X_test, y_test)
print()
print(f"Accuracy {accuracy}")

y_proba = stacking_model.predict_proba(X_test)

roc_auc = roc_auc_score(
    y_test,
    y_proba,
    multi_class="ovr",
    average="macro"
)

print(f"ROC-AUC (OvR, macro): {roc_auc:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001922 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 49401, number of used features: 25
[LightGBM] [Info] Start training from score -1.268082
[LightGBM] [Info] Start training from score -0.647312
[LightGBM] [Info] Start training from score -1.633842

Accuracy 0.860213747874666
ROC-AUC (OvR, macro): 0.9143


In [9]:
# pretty okay result, but not better than one of the models in benchmarking